# thui-a3-wm-smoke (Thuitanium / Knowless Crew) — A3: an executable world model, kept and verified

**This is a Knowless Crew / Thuitanium experiment notebook.** It is `thui-animfast-b71-full25-r1` with one change in
cell 13: the agent may save a Python world model (`WORLD_MODEL = '''...'''`) that is loaded at the top of every later
python call, and `verify_world_model()` replays recorded transitions through its `predict(rows, action)`. A short block
appended to each user prompt explains this and reports the model's status. Serving profile, solver and clock unchanged.
Smoke: 3 games at 1800 s.

Serving stack by [Keith Tyser](https://www.kaggle.com/code/keithtyser/duck-qwen3-8-flash-next-nvfp4-mtp), harness by
[Tufa Labs](https://www.kaggle.com/code/jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner), anim solver
bundle `jakobbrggen/taaf-kaggle-source-anim-20260807-anim`. Idea after arXiv 2605.05138 / 2607.15439 and NIMI's Tycho.


## Upstream notes — the Tufa Labs duck harness (their text, reworded in the third person)

The duck harness notebook this fork descends from is Tufa Labs' "duck harness" (their June 30 milestone
winner). Their own note on it: the readable notebook scored Tufa Labs' milestone-winning 1.21, and later
runs of it did not repeat that result; the original, less readable notebook is also shared at
https://www.kaggle.com/code/jeroencottaar/taaf-duck-harness-kaggle and is not recommended.

- Tufa Labs' writeup of what the solver does: https://www.kaggle.com/competitions/arc-prize-2026-arc-agi-3/discussion/717133
- Machine Learning Street Talk interview by Tim Scarfe about the duck harness: https://x.com/MLStreetTalk/status/2072326433922297975?s=20

The solver was written by the Tufa Labs team; in alphabetical order: Harold Bessis, Jeroen Cottaar,
Isaiah Pressman, Andries Smit, Michal Tesnar, and Stefano Viel. The notebook holds infrastructure and
diagnostics only; the solver code lives in the attached source bundle. It installs the ARC runtime from the
competition wheelhouse, makes the bundled source snapshot importable, runs the solver setup commands, loads
the pickled benchmark, plays the competition games, and writes results to `/kaggle/working`. Diagnostics are
minimised during a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`) and kept full otherwise. A copy of
this notebook must select the RTX Pro 6000 GPU manually.


## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
# thui-animfast: full diagnostics on an interactive public run (usage/events/transcript sidecars); minimal in a rerun.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

## 2. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# thui-animfast: resolve the competition mount instead of assuming its layout -- Kaggle serves either
# /kaggle/input/competitions/<comp> or /kaggle/input/<comp>, and which one varies between runs.
_COMP_CANDIDATES = ["/kaggle/input/competitions/arc-prize-2026-arc-agi-3", "/kaggle/input/arc-prize-2026-arc-agi-3"]
_COMP_DIR = next((_p for _p in _COMP_CANDIDATES if os.path.isdir(_p)), None)
assert _COMP_DIR is not None, (
    "thui-animfast: no competition mount found. Tried " + repr(_COMP_CANDIDATES)
    + "; /kaggle/input holds "
    + repr(sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else "MISSING")
)
_WHEELS = os.path.join(_COMP_DIR, "arc_agi_3_wheels")
assert os.path.isdir(_WHEELS), "thui-animfast: resolved wheels dir is not a directory: " + _WHEELS
print("thui-animfast: competition mount = " + _COMP_DIR, flush=True)
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        _WHEELS,
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 3. Locate the source bundle

Find the uploaded TAAF source dataset by its marker file, and record where Kaggle mounted
every attached input so setup commands and the solver can find them.

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["keithtyser/duck-qwen38-nvfp4-mtp-vllm-smoke-v1", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1", "jakobbrggen/taaf-kaggle-source-anim-20260807-anim"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir(label: str) -> Path:
    # thui-animfast: TWO attached datasets carry the marker (his June duck bundle and the anim bundle), so
    # "first marker wins" is a coin flip -- pick by the benchmark_label the marker file records.
    found = {}
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        try:
            found[json.loads(marker.read_text())["benchmark_label"]] = marker.parent
        except Exception as exc:
            print(f"thui-animfast: unreadable marker {marker}: {exc!r}", flush=True)
    if label not in found:
        raise RuntimeError(f"TAAF source bundle {label!r} not found under /kaggle/input; markers = {found}")
    return found[label]


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir("duck-harness-kaggle")          # his: serving_setup.py, vllm patches, watchdog, teardown
ANIM_BUNDLE_DIR = _find_bundle_dir("anim-20260807-anim")     # ours: the solver tree + its pickled benchmark / target
assert BUNDLE_DIR != ANIM_BUNDLE_DIR, "thui-animfast: both labels resolved to one directory"
assert (BUNDLE_DIR / "serving_setup.py").is_file(), f"thui-animfast: his bundle has no serving_setup.py: {BUNDLE_DIR}"
assert (ANIM_BUNDLE_DIR / "src" / "ARC3-Inference" / "inference" / "utils" / "animation.py").is_file(), (
    f"thui-animfast: the anim bundle has no animation.py: {ANIM_BUNDLE_DIR}")
print(f"thui-animfast: anim bundle = {ANIM_BUNDLE_DIR}", flush=True)
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

## 4. Import the bundled source and run solver setup

Put the snapshotted repositories on the path (this process and any child processes), then run
the solver's setup commands â€” installing wheels, fetching model weights, and so on.

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
# thui-animfast: his tree minus the two solver repos (the June duck), plus the anim solver tree. The loop below
# inserts each entry at sys.path[0], so the LAST entries win -- the anim ones; the .pth is written anim-first.
_SOLVER_REPOS = {"ARC3-Inference", "tufa-arc-agi-framework"}
_his_entries = [e for e in _source_path_entries(BUNDLE_DIR) if e.parent.name not in _SOLVER_REPOS and e.name not in _SOLVER_REPOS]
_anim_entries = _source_path_entries(ANIM_BUNDLE_DIR)
assert _anim_entries and all(str(e).startswith(str(ANIM_BUNDLE_DIR)) for e in _anim_entries), _anim_entries
assert not any(("ARC3-Inference" in str(e) or "tufa-arc-agi-framework" in str(e)) for e in _his_entries), _his_entries
source_entries = _his_entries + _anim_entries
print(f"thui-animfast: source roots his={[str(e) for e in _his_entries]} anim={[str(e) for e in _anim_entries]}", flush=True)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in (_anim_entries + _his_entries)))   # anim first for child processes
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
env = _command_env()
for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
    print(f"taaf.kaggle: setup command: {command}", flush=True)
    subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
    # Re-read in case the command persisted new env keys.
    env = _command_env()
    os.environ.update(env)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)
# ---- thui-animfast: the thui-v3 knobs, set AFTER his serving_setup persisted the analyzer env and BEFORE any
# `inference` import (tool_agent reads LOCAL_ANALYZER_SEED / YIELD_SECONDS at import time), then the graft teeth.
_KNOBS = {"LOCAL_ANALYZER_SEED": "20260825", "LOCAL_ANALYZER_YIELD_SECONDS": "180"}
_persisted = json.loads(SETUP_ENV_PATH.read_text())
assert _persisted.get("LOCAL_ANALYZER_MODEL_ID") == "Qwen/Qwen3.8-Flash-Next-NVFP4", _persisted.get("LOCAL_ANALYZER_MODEL_ID")
assert _persisted.get("LOCAL_ANALYZER_YIELD_SECONDS") == "60", "his serving_setup no longer persists yield 60 -- re-derive the override"
assert _persisted.get("LOCAL_ANALYZER_TEMPERATURE") == "0.6" and _persisted.get("MULTIMODAL_UPSCALE") == "4", _persisted
_persisted.update(_KNOBS)
SETUP_ENV_PATH.write_text(json.dumps(_persisted, indent=2, sort_keys=True) + "\n")
os.environ.update(_KNOBS)
assert "inference" not in sys.modules and "taaf" not in sys.modules, "solver imported before the knob override"
import inference.agent.tool_agent as _tool_agent
import inference.utils.animation as _anim_mod
import taaf as _taaf
for _m in (_tool_agent, _anim_mod, _taaf):
    assert str(Path(_m.__file__).resolve()).startswith(str(ANIM_BUNDLE_DIR.resolve())), (_m.__name__, _m.__file__)
assert _tool_agent._LOCAL_ANALYZER_SEED == int("20260825"), _tool_agent._LOCAL_ANALYZER_SEED
assert float(_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS) == float("180"), _tool_agent._LOCAL_ANALYZER_YIELD_SECONDS
assert os.environ["LOCAL_ANALYZER_MODEL_ID"] == "Qwen/Qwen3.8-Flash-Next-NVFP4"
print(f"THUI_ANIMFAST_GRAFT ok solver={Path(_tool_agent.__file__).parent} seed={_tool_agent._LOCAL_ANALYZER_SEED} "
      f"yield={_tool_agent._LOCAL_ANALYZER_YIELD_SECONDS} model={os.environ['LOCAL_ANALYZER_MODEL_ID']} "
      f"temperature={os.environ['LOCAL_ANALYZER_TEMPERATURE']} upscale={os.environ['MULTIMODAL_UPSCALE']}", flush=True)


## 5. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(ANIM_BUNDLE_DIR / "deploy_target.pkl", "rb") as file:   # thui-animfast: the anim bundle's target (32400 s)
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(ANIM_BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:   # thui-animfast: the anim solver
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR
# thui-animfast: the unpickled solver must be the anim chassis, not the June duck.
assert bm.label == "anim-20260807-anim", bm.label
assert getattr(bm.solver, "animation_awareness", None) is True and getattr(bm.solver, "hard_noop_guard", None) is True, vars(bm.solver)
assert type(bm.solver).__module__ == "inference.framework.solver"
print(f"thui-animfast: bm.label={bm.label} solver={type(bm.solver).__name__} animation_awareness={bm.solver.animation_awareness} "
      f"hard_noop_guard={bm.solver.hard_noop_guard} target.max_runtime_s={target.max_runtime_s}", flush=True)


## 6. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts â€” the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Exact public-25 and competition settings.
bm.solver.max_runtime_s_per_game = 7920.0
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 28
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = False
if float(getattr(target, 'max_runtime_s', 0.0) or 0.0) != 32400.0:
    raise RuntimeError(
        f'Expected the 32400-second notebook budget, got {target.max_runtime_s!r}.'
    )
print(
    f'PUBLIC25_SETTINGS budget_s={bm.solver.max_runtime_s_per_game} '
    f'concurrency={bm.solver.concurrency} analyzer_timeout={bm.solver.analyzer_timeout} '
    f'action_cap={bm.solver.max_actions_per_game} request_logs={bm.solver.save_request_logs}',
    flush=True,
)

# ---- thui-a3 (A3): an executable world model the agent writes, keeps, and checks against what happened ----
# The python tool is ephemeral (one sandbox subprocess per call), so nothing the model writes survives a call. This graft
# (1) stores the string assigned to a top-level `WORLD_MODEL = '''...'''` in any python call, per game session;
# (2) prepends that source, plus parse_action() and verify_world_model(), to every later python call, so predict()
#     and the agent's other functions are simply defined; the prefix costs no prompt tokens;
# (3) verify_world_model() replays recorded same-level transitions through predict(rows, action) and reports misses;
# (4) appends a short block to every user prompt: the instruction and the current model / verify status.
# Unchanged: the harness, the solver, the serving profile; no action is issued or refused by this code.
import ast as _a3_ast
import re as _a3_re
import textwrap as _a3_textwrap
import threading as _a3_threading

import inference.agent.tool_agent as _a3_ta

_A3_TL = _a3_threading.local()
_A3_MAX_SRC_CHARS = 16000
_A3_STATS = {"python_calls": 0, "calls_with_model": 0, "saves": 0, "rejects": 0, "verifies": 0, "load_errors": 0, "prompts": 0}

_A3_HELPERS = r'''
def parse_action(action):
    import re as _re
    text = str(action)
    name = _re.match(r"\s*([A-Za-z_][A-Za-z0-9_]*)", text)
    row = _re.search(r"row\s*=\s*(-?\d+)", text)
    col = _re.search(r"col\s*=\s*(-?\d+)", text)
    return (name.group(1) if name else text, int(row.group(1)) if row else None, int(col.group(1)) if col else None)


def verify_world_model(show=3, last=60):
    try:
        _predict = predict
    except Exception:
        print("A3_VERIFY no predict(rows, action) is defined -- save WORLD_MODEL first")
        return None
    pairs = [t for t in transitions if t.before_frame is not None and t.after_frame is not None
             and t.before_frame.level == t.after_frame.level]
    pairs = pairs[-last:]
    exact = 0
    raised = 0
    bad = []
    for t in pairs:
        before = str(t.before_frame.ascii).split("\n")
        after = str(t.after_frame.ascii).split("\n")
        try:
            pred = [str(r) for r in _predict(list(before), t.action)]
        except Exception as exc:
            raised += 1
            if len(bad) < show:
                bad.append((t.action, "raised " + type(exc).__name__ + ": " + str(exc)[:100]))
            continue
        if pred == after:
            exact += 1
            continue
        wrong = []
        for r in range(max(len(after), len(pred))):
            a_row = after[r] if r < len(after) else ""
            p_row = pred[r] if r < len(pred) else ""
            for c in range(max(len(a_row), len(p_row))):
                a = a_row[c] if c < len(a_row) else "?"
                p = p_row[c] if c < len(p_row) else "?"
                if a != p:
                    wrong.append((r, c, p, a))
        if len(bad) < show:
            bad.append((t.action, str(len(wrong)) + " cells wrong; first (row, col, predicted, actual): " + str(wrong[:4])))
    print("A3_VERIFY exact=" + str(exact) + "/" + str(len(pairs)) + " raised=" + str(raised))
    for item in bad:
        print("  miss:", item[0], "->", item[1])
    return {"exact": exact, "total": len(pairs), "raised": raised}
'''


def _a3_state(agent):
    run_dir = getattr(agent, "_session_runtime_dir", None)
    st = getattr(agent, "_a3_state", None)
    if not isinstance(st, dict) or st.get("dir") != run_dir:
        st = {"dir": run_dir, "src": "", "saves": 0, "last_reject": "", "verify": "", "load_error": ""}
        agent._a3_state = st
    return st


def _a3_extract(code):
    try:
        tree = _a3_ast.parse(code)
    except SyntaxError:
        return None
    for node in tree.body:
        if (isinstance(node, _a3_ast.Assign) and len(node.targets) == 1 and isinstance(node.targets[0], _a3_ast.Name)
                and node.targets[0].id == "WORLD_MODEL" and isinstance(node.value, _a3_ast.Constant)
                and isinstance(node.value.value, str)):
            return node.value.value
    return None


def _a3_prefix(src):
    parts = [_A3_HELPERS]
    if src:
        parts.append("try:\n" + _a3_textwrap.indent(src, "    ") + "\n    pass\n"
                     "except Exception as _a3_exc:\n"
                     "    print('A3_MODEL_LOAD_ERROR ' + type(_a3_exc).__name__ + ': ' + str(_a3_exc)[:160])\n")
    return "\n".join(parts) + "\n"


_a3_orig_run_python_tool = _a3_ta.ToolAgent._run_python_tool
_a3_orig_sandbox = _a3_ta.run_sandboxed_python
_a3_orig_build_user_prompt = _a3_ta.ToolAgent._build_user_prompt


def _a3_run_python_tool(self, state_path, arguments):
    try:
        self._ensure_session(state_path)
        st = _a3_state(self)
        new_src = _a3_extract(str((arguments or {}).get("code", "")))
        if new_src is not None:
            if len(new_src) > _A3_MAX_SRC_CHARS:
                st["last_reject"] = "too long (" + str(len(new_src)) + " chars > " + str(_A3_MAX_SRC_CHARS) + ")"
                _A3_STATS["rejects"] += 1
            else:
                try:
                    compile(new_src, "<world_model>", "exec")
                    st["src"], st["last_reject"], st["verify"], st["load_error"] = new_src, "", "", ""
                    st["saves"] += 1
                    _A3_STATS["saves"] += 1
                except SyntaxError as exc:
                    st["last_reject"] = "SyntaxError line " + str(exc.lineno) + ": " + str(exc.msg)
                    _A3_STATS["rejects"] += 1
    except Exception as exc:   # the graft must never cost the turn
        print(f"thui-a3: store error (pass-through): {type(exc).__name__}: {exc}", flush=True)
    _A3_TL.agent = self
    try:
        return _a3_orig_run_python_tool(self, state_path, arguments)
    finally:
        _A3_TL.agent = None


def _a3_sandbox(*, code, **kwargs):
    agent = getattr(_A3_TL, "agent", None)
    if agent is None:
        return _a3_orig_sandbox(code=code, **kwargs)
    st = _a3_state(agent)
    prefix = _a3_prefix(st["src"])
    offset = prefix.count("\n")
    _A3_STATS["python_calls"] += 1
    _A3_STATS["calls_with_model"] += 1 if st["src"] else 0
    result = _a3_orig_sandbox(code=prefix + code, **kwargs)
    try:
        err = result.get("error")
        if isinstance(err, str) and err:
            def _fix(m):
                n = int(m.group(1)) - offset
                return 'File "<python_tool>", line ' + (str(n) if n > 0 else "(saved WORLD_MODEL / helpers)")
            result["error"] = _a3_re.sub(r'File "<python_tool>", line (\d+)', _fix, err)
        out = str(result.get("stdout") or "")
        hits = _a3_re.findall(r"A3_VERIFY exact=(\d+)/(\d+) raised=(\d+)", out)
        if hits:
            e, t, r = hits[-1]
            st["verify"] = e + "/" + t + " exact" + (", " + r + " raised" if r != "0" else "")
            _A3_STATS["verifies"] += 1
        load = _a3_re.findall(r"A3_MODEL_LOAD_ERROR ([^\n]*)", out)
        if load:
            st["load_error"] = load[-1][:160]
            _A3_STATS["load_errors"] += 1
        if _A3_STATS["python_calls"] % 200 == 0:
            print(f"thui-a3: STATS {_A3_STATS}", flush=True)
    except Exception as exc:
        print(f"thui-a3: post error (pass-through): {type(exc).__name__}: {exc}", flush=True)
    return result


_A3_PROMPT = (
    "\n\n[EXECUTABLE WORLD MODEL]\n"
    "Keep your best hypothesis of this game's rules as runnable code. In any python call, assign "
    "WORLD_MODEL = '''<python source>''' as a plain string literal at top level. The source must define "
    "predict(rows, action) -> rows, where rows is a frame as a list of ASCII strings (frame.ascii.split('\\n')) and "
    "action is a transition's action string (parse_action(action) returns name, row, col). Once saved, the source runs "
    "at the top of every later python call, so predict and anything else it defines is already there. "
    "verify_world_model() replays the recorded transitions through predict and prints the ones it gets wrong. "
    "When it misses, fix the model before trusting a multi-step plan; when it matches, simulate candidate action "
    "sequences with predict before spending real actions.\n"
)


def _a3_build_user_prompt(self, *args, **kwargs):
    text = _a3_orig_build_user_prompt(self, *args, **kwargs)
    try:
        st = _a3_state(self)
        if st["src"]:
            status = "saved (" + str(st["src"].count("\n") + 1) + " lines, save #" + str(st["saves"]) + ")"
            status += "; last verify: " + (st["verify"] or "not run since this save")
            if st["load_error"]:
                status += "; LOAD ERROR: " + st["load_error"]
        else:
            status = "none saved yet"
        if st["last_reject"]:
            status += "; last WORLD_MODEL rejected: " + st["last_reject"]
        _A3_STATS["prompts"] += 1
        return text + _A3_PROMPT + "World model status: " + status + "\n"
    except Exception as exc:
        print(f"thui-a3: prompt error (pass-through): {type(exc).__name__}: {exc}", flush=True)
        return text

# ---- thui-a3 teeth: run BEFORE install, against the REAL sandbox (in-kernel: the bundle's; locally: the same file) ----
def _a3_teeth():
    class _FakeAgent:
        _session_runtime_dir = "teeth-session"
        def _ensure_session(self, state_path):
            pass

    def _frame(rows, level=1, step=0):
        return {"ascii": "\n".join(rows), "step": step, "level": level, "shape": [len(rows), len(rows[0])],
                "grid": [[0] * len(rows[0]) for _ in rows]}

    f0 = ["ab", "cd"]
    f2 = ["ab", "cX"]
    state = {"current_frame": _frame(f2, step=2),
             "history": [{"action": "", "frame": _frame(f0)}, {"action": "UP", "frame": _frame(f0, step=1)},
                         {"action": "MOUSE(row=1, col=1)", "frame": _frame(f2, step=2)}],
             "valid_actions": ["UP"], "last_action_result": {}}

    def _run(agent, code):
        _A3_TL.agent = agent
        try:
            return _a3_sandbox(code=code, timeout_seconds=20, initial_state=state,
                               action_handler=lambda actions: {}, animation_handler=None)
        finally:
            _A3_TL.agent = None

    # 1. extraction: only a top-level plain string literal counts
    assert _a3_extract("WORLD_MODEL = '''def predict(rows, action):\n    return rows\n'''\n").startswith("def predict")
    assert _a3_extract("WORLD_MODEL = 'a' + 'b'\n") is None
    assert _a3_extract("def f():\n    WORLD_MODEL = 'x'\n") is None
    assert _a3_extract("WORLD_MODEL = f'{1}'\n") is None

    # 2. store + real sandbox + verify: identity predict gets the no-change step and misses the click
    agent = _FakeAgent()
    st = _a3_state(agent)
    st["src"] = "def predict(rows, action):\n    return rows\n"
    res = _run(agent, "out = verify_world_model()\nprint(parse_action('MOUSE(row=1, col=1)'))\n")
    out = str(res.get("stdout") or "")
    assert "A3_VERIFY exact=1/2 raised=0" in out, ("verify count", res)
    assert "(0, 1, 'd', 'X')" in out or "(1, 1, 'd', 'X')" in out, ("miss cell", out)
    assert "('MOUSE', 1, 1)" in out, ("parse_action", out)
    assert st["verify"] == "1/2 exact", st

    # 3. a wrong model that raises is counted, not fatal
    st["src"] = "def predict(rows, action):\n    raise ValueError('nope')\n"
    out = str(_run(agent, "verify_world_model()\n").get("stdout") or "")
    assert "A3_VERIFY exact=0/2 raised=2" in out, out

    # 4. user error line numbers are the user's own
    err = str(_run(agent, "x = 1\nraise ValueError('boom')\n").get("error") or "")
    assert 'File "<python_tool>", line 2' in err, err

    # 5. a model that fails at load is reported and the user's code still runs
    st["src"] = "raise RuntimeError('bad model')\n"
    res = _run(agent, "print('after-load')\n")
    out = str(res.get("stdout") or "")
    assert "A3_MODEL_LOAD_ERROR RuntimeError: bad model" in out and "after-load" in out, res
    assert st["load_error"].startswith("RuntimeError"), st

    # 6. control: with no agent bound (another thread, or the graft not entered) the code runs untouched
    res = _a3_sandbox(code="print('verify_world_model' in dir())\n", timeout_seconds=20, initial_state=state,
                      action_handler=lambda actions: {}, animation_handler=None)
    assert str(res.get("stdout") or "").strip() == "False", res

    # 7. prompt block carries the instruction and the live status
    class _PromptAgent(_FakeAgent):
        pass
    pa = _PromptAgent()
    global _a3_orig_build_user_prompt
    saved = _a3_orig_build_user_prompt
    _a3_orig_build_user_prompt = lambda self, *a, **k: "BASE"
    try:
        text = _a3_build_user_prompt(pa, 1, valid_actions=["UP"])
        assert text.startswith("BASE") and "[EXECUTABLE WORLD MODEL]" in text and "none saved yet" in text, text
        _a3_state(pa)["src"] = "def predict(rows, action):\n    return rows\n"
        _a3_state(pa)["saves"] = 1
        assert "saved (3 lines, save #1)" in _a3_build_user_prompt(pa, 2, valid_actions=["UP"])
    finally:
        _a3_orig_build_user_prompt = saved
    for k in _A3_STATS:
        _A3_STATS[k] = 0
    print("thui-a3: teeth ok (7 checks, real sandbox)", flush=True)


_a3_teeth()

# ---- thui-a3 install (after the teeth) ----
import inspect as _a3_inspect
assert "run_sandboxed_python(" in _a3_inspect.getsource(_a3_orig_run_python_tool), "the tool no longer calls the module-level sandbox"
assert "def _build_user_prompt" in _a3_inspect.getsource(_a3_ta.ToolAgent)
_a3_ta.ToolAgent._run_python_tool = _a3_run_python_tool
_a3_ta.run_sandboxed_python = _a3_sandbox
_a3_ta.ToolAgent._build_user_prompt = _a3_build_user_prompt
assert _a3_ta.ToolAgent._run_python_tool is _a3_run_python_tool and _a3_ta.run_sandboxed_python is _a3_sandbox
assert _a3_ta.ToolAgent._build_user_prompt is _a3_build_user_prompt
print(f"THUI_A3_GRAFT ok tool_agent={_a3_ta.__file__} max_src={_A3_MAX_SRC_CHARS}", flush=True)


## 7. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise â€” an interactive "Save & Run" â€” play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Skip pre-run display and git-status copies; the staged identity pins the run.

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

PUBLIC_GAME_IDS = tuple(['tn36-ef4dde99', 'vc33-5430563c', 'bp35-0a0ad940'])   # thui-a3 smoke subset

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path(_COMP_DIR) / "environment_files")   # thui-animfast: resolved in cell 5
    offline_games = _offline_games(competition_env_files)
    offline_by_id = {game.env_name: game for game in offline_games}
    if len(offline_by_id) != len(offline_games):
        raise RuntimeError('The offline public game list contains duplicate IDs.')
    missing = sorted(set(PUBLIC_GAME_IDS) - set(offline_by_id))
    extra = sorted(set(offline_by_id) - set(PUBLIC_GAME_IDS))
    if missing or (extra and len(PUBLIC_GAME_IDS) == 25):   # thui-a3 smoke: a subset leaves extras by design
        raise RuntimeError(
            f'Offline public game set changed; missing={missing}, extra={extra}.'
        )
    bm.games = [offline_by_id[game_id] for game_id in PUBLIC_GAME_IDS]
    bm.solver.max_runtime_s_per_game = 1800.0   # thui-a3 smoke clock
    print(f"thui-a3: smoke {len(bm.games)} games @ {bm.solver.max_runtime_s_per_game} s", flush=True)
    if len(bm.games) != len(PUBLIC_GAME_IDS):
        raise RuntimeError(f'Expected 25 public games, got {len(bm.games)}.')
    print(f'PUBLIC25_SELECTION games={len(bm.games)} passes=1', flush=True)

bm.n_passes = 1
bm.game_weights = None

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
if budget <= 600.0:
    raise RuntimeError(f'Notebook budget is too small for the teardown reserve: {budget}.')
soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(
    seconds=budget - 600.0
)

# Start recovery only after setup readiness and all run gates pass.
if str(BUNDLE_DIR) not in sys.path:
    sys.path.insert(0, str(BUNDLE_DIR))
import vllm_server_watchdog as vllm_watchdog

vllm_watchdog_setup = vllm_watchdog.load_setup(BUNDLE_DIR / 'serving_setup.py')
vllm_watchdog.start_background(
    vllm_watchdog_setup,
    vllm_watchdog.WatchdogConfig(
        interval_seconds=15.0,
        request_timeout_seconds=5,
        failure_threshold=4,
        max_restart_attempts=2,
    ),
)

# Play the benchmark; watchdog stop and teardown run even if it raises.
try:
    await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=True)
    bm._save_json()
    if not TRUE_SUBMISSION:
        # Kaggle Save & Run expects this valid placeholder after an offline run.
        # A real competition rerun uses the live gateway and never enters this branch.
        import pandas as pd

        pd.DataFrame(
            [["1_0", "1", True, 1]],
            columns=["row_id", "game_id", "end_of_game", "score"],
        ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

        # Check terminal coverage, then call the frozen scorer once.
        # This path does not render HTML.
        public_runs = list(bm.game_runs)
        public_run_ids = [run.game_id for run in public_runs]
        if len(public_runs) != len(PUBLIC_GAME_IDS) or public_run_ids != list(PUBLIC_GAME_IDS):
            raise RuntimeError(
                f'Public run coverage changed: count={len(public_runs)} ids={public_run_ids}.'
            )
        unfinished = [
            (run.game_id, run.state, run.final_score)
            for run in public_runs
            if run.state not in {'won', 'gave_up', 'cancelled'}
            or run.final_score is None
        ]
        if unfinished:
            raise RuntimeError(f'Public runs did not finalize cleanly: {unfinished}.')
        crashed = [run.game_id for run in public_runs if run.state == 'crashed']
        if crashed:
            raise RuntimeError(f'Public runs crashed: {crashed}.')
        total_actions = sum(len(run.history) for run in public_runs)
        if total_actions <= 0:
            raise RuntimeError('Public runs produced no actions.')

        from inference.tools.eval import evaluate_runs, save_score_file

        score_summary = evaluate_runs([WORKING_DIR])
        score_path = save_score_file(
            score_summary,
            run_dirs=[WORKING_DIR],
            output_path=WORKING_DIR / "score.json",
        )
        if Path(score_path) != WORKING_DIR / 'score.json' or not Path(score_path).is_file():
            raise RuntimeError(f'Frozen scorer did not write score.json: {score_path}.')
        print(
            f'PUBLIC25_AUDIT runs=25 actions={total_actions} score_path={score_path}',
            flush=True,
        )
finally:
    try:
        vllm_watchdog.stop_background(timeout_seconds=15.0)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            print(f"taaf.kaggle: teardown command: {command}", flush=True)
            subprocess.run(
                command,
                shell=True,
                check=False,
                cwd=WORKING_DIR,
                env=_command_env(),
                timeout=30.0,
            )


## 8. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
# Minimal diagnostics are enabled; skip post-run HTML rendering.
